# ISIC 2019 — Stage 1: Kaggle setup and dataset verification

**Course:** AI for Trustworthy Decision Making  
**Purpose:** Inspect the official ISIC 2019 training images, diagnostic labels, and metadata before building centralized and federated models.

**Before running:** Open this notebook in Kaggle, attach an ISIC 2019 dataset using **Add Input**, and leave **Accelerator = None**. The attached dataset must contain the **2019** training-label and training-metadata CSV files (not the ISIC 2020 binary competition).

This notebook does **not** train any model or download a 9 GB ZIP. It verifies data access, identifies missing images, and saves a class-distribution chart. Run cells from top to bottom.

## 1. Check the runtime

In [ ]:
from pathlib import Path
import platform
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed (not required for dataset checks).")

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)


## 2. Look at Kaggle input files

Attached datasets appear under `/kaggle/input/`. This directory is read-only; save figures and model checkpoints to `/kaggle/working/`.

In [ ]:
INPUT_DIR = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')

if not INPUT_DIR.exists():
    raise FileNotFoundError('Not running in Kaggle, or /kaggle/input is missing.')

folders = [p for p in INPUT_DIR.iterdir() if p.is_dir()]
print('Attached input datasets:', [p.name for p in folders])
if not folders:
    raise RuntimeError('No dataset attached. Use the right panel: Add Input → choose ISIC 2019.')

csvs = sorted(INPUT_DIR.rglob('*.csv'))
zips = sorted(INPUT_DIR.rglob('*.zip'))
print('\nCSV files:')
for p in csvs[:30]: print(' -', p)
print('\nZIP files:')
for p in zips[:10]: print(' -', p)


## 3. Load official ISIC 2019 labels and metadata

The files should have the exact official names below. This prevents us from accidentally loading ISIC 2020 or another dataset.

In [ ]:
def one_file(filename):
    matches = list(INPUT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one {filename}, found {len(matches)}. '
            'Check attached datasets and folder names. Matches: ' + str(matches[:5])
        )
    return matches[0]

labels_file = one_file('ISIC_2019_Training_GroundTruth.csv')
metadata_file = one_file('ISIC_2019_Training_Metadata.csv')
labels = pd.read_csv(labels_file)
metadata = pd.read_csv(metadata_file)

CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
required_label_columns = {'image', *CLASSES}
if not required_label_columns.issubset(labels.columns):
    raise ValueError('Unexpected label CSV columns: ' + str(labels.columns.tolist()))
if 'image' not in metadata.columns:
    raise ValueError('Metadata CSV has no image column.')

print('Labels rows:', len(labels), '(expected 25,331)')
print('Metadata rows:', len(metadata), '(expected 25,331)')
print('Label columns:', labels.columns.tolist())
print('Metadata columns:', metadata.columns.tolist())
print('Duplicate label image IDs:', labels['image'].duplicated().sum())
print('Duplicate metadata image IDs:', metadata['image'].duplicated().sum())
print('Rows without exactly one of eight labels:', (labels[CLASSES].sum(axis=1) != 1).sum())
assert len(labels) == 25331, 'Unexpected training-label count. Check dataset version.'
assert len(metadata) == 25331, 'Unexpected training-metadata count. Check dataset version.'
assert (labels[CLASSES].sum(axis=1) == 1).all(), 'Some images do not have exactly one class.'


## 4. Verify the image files

Kaggle may unpack the archive into JPEG files while importing. If it leaves a ZIP, this cell reads the ZIP *directory only* instead of extracting gigabytes into the temporary workspace.

In [ ]:
from zipfile import ZipFile

image_files = [p for p in INPUT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg'}]
print('Visible JPEG files:', len(image_files))

if image_files:
    image_ids = {p.stem for p in image_files}
    image_source = 'unpacked JPEG files'
else:
    zip_candidates = [p for p in zips if '2019' in p.name and 'Training_Input' in p.name]
    if not zip_candidates:
        print('No JPEG images or official training ZIP found. Attach your image dataset.')
        image_ids = set()
        image_source = 'none'
    else:
        with ZipFile(zip_candidates[0]) as archive:
            jpeg_names = [name for name in archive.namelist() if name.lower().endswith(('.jpg','.jpeg'))]
        image_ids = {Path(name).stem for name in jpeg_names}
        image_source = f'ZIP index: {zip_candidates[0].name}'

expected_ids = set(labels['image'])
missing_ids = sorted(expected_ids - image_ids)
extra_ids = sorted(image_ids - expected_ids)
print('Source:', image_source)
print('Distinct training image IDs found:', len(image_ids))
print('Missing training images:', len(missing_ids))
print('Example missing IDs:', missing_ids[:5])
print('Other image IDs (e.g. test data):', len(extra_ids))

if not missing_ids:
    print('SUCCESS: every labeled training image was located!')
else:
    print('ACTION NEEDED: confirm the complete ISIC 2019 training images are attached.')


## 5. Inspect class balance and save a plot

This matters because a model can achieve high ordinary accuracy by favoring frequent lesion classes. Later we will report balanced accuracy and macro F1 as well.

In [ ]:
class_counts = labels[CLASSES].sum().astype(int).sort_values(ascending=False)
print('Images per diagnosis:')
print(class_counts.to_string())

class_counts.rename('image_count').to_csv(OUTPUT_DIR / 'isic2019_class_counts.csv')
fig, ax = plt.subplots(figsize=(9, 4.5))
class_counts.plot(kind='bar', ax=ax)
ax.set_title('ISIC 2019 — Training image distribution by class')
ax.set_xlabel('Diagnosis class')
ax.set_ylabel('Images')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'isic2019_class_distribution.png', dpi=160)
plt.show()
print('Saved output files in:', OUTPUT_DIR)


## 6. Check metadata and lesion identifiers

In [ ]:
combined = labels.merge(metadata, on='image', how='left', validate='one_to_one')
combined['label'] = combined[CLASSES].idxmax(axis=1)
print('Combined rows:', len(combined))
print('Missing values by metadata column:')
for col in metadata.columns:
    if col != 'image':
        print(f' - {col}: {combined[col].isna().sum()}')
if 'lesion_id' in combined.columns:
    print('Unique non-missing lesion identifiers:', combined['lesion_id'].nunique())
    print('Images with missing lesion identifier:', combined['lesion_id'].isna().sum())
else:
    print('Warning: no lesion_id column. Splitting requires extra care to avoid leakage.')
print('\nNext: build reproducible lesion-aware train/validation/test splits.')


## What comes next?

After the checks succeed:

1. Create a reproducible training / validation / held-out test split that keeps the same known lesion in **one** split.
2. Load images with PyTorch, resize to 224×224, and apply the pretrained ResNet18 preprocessing.
3. Train one **centralized ResNet18** baseline.
4. Simulate **three non-IID federated clients** with Flower and FedAvg.
5. Evaluate both versions on exactly the same held-out test set. Then proceed to trustworthiness enhancements.

**Note:** Your official ISIC 2019 test set is not needed for these initial checks. It can be reserved for final evaluation under an explicitly documented protocol. If your uploaded files are ZIP-only, decide how to stream or repackage images for training rather than extracting blindly into limited notebook storage.
